In [ ]:
# ==============================================================================
# PIPELINE UNIFIED: LIQUIDITY DETECTION (PDH/PDL/PWH/PWL) - TIMEFRAME H1
# VERSI 6.0 - Satu script, satu tabel gold (xauusd_liquidity_gold)
# ==============================================================================
#
# TUJUAN
#   Menggabungkan Script Daily (v3.1, PDH/PDL) dan Weekly (v5.0, PWH/PWL) menjadi
#   SATU pipeline sekali jalan. H1 dimuat sekali, lead dihitung sekali, helper
#   pelabelan yang SAMA dipanggil 4x (PDH, PDL, PWH, PWL), lalu di-union ke satu
#   tabel gold `market.default.xauusd_liquidity_gold` dengan kolom `level_type`.
#   Ini juga menutup mismatch lama: Daily/Weekly menulis dua tabel terpisah,
#   sementara Script 3 (D2-D7 analytics) sudah membaca tabel unified ini.
#
#   LOGIC/ATURAN IDENTIK dengan existing. Tidak ada aturan baru.
#
# TEMUAN KRITIS YANG DIPERTAHANKAN (T1, T2, T3, T4, T6, T12, T12c, T13)
#   T1 : 1 kolom `outcome` 4 kelas saling lepas (bukan 3 kolom target redundan).
#        Cabang terakhir sengaja NULL sebagai sentinel, lalu di-assert nol.
#   T2 : buang jendela tidak lengkap (close_lead_N NULL) - WAJIB SETELAH exhaustion.
#   T3 : lead() mengambil BARIS berikutnya, bukan JAM. Jendela bisa menembus akhir
#        pekan -> ditandai window_is_continuous (PENANDA, bukan filter).
#   T4 : label sesi sadar zona waktu (from_utc_timestamp), bukan offset tetap.
#   T6 : LEFT join level (bukan inner) -> orphan tetap tercatat di funnel.
#   T12: kunci minggu = date_trunc("week", session_date) tunggal & monoton.
#   T12c: candle Minggu (pembuka pekan) terlabeli sesi Senin lewat session_date.
#   T13: batas hari = 17:00 New York (bukan kalender UTC). Level D1/W-1 diagregasi
#        ULANG dari H1, TIDAK memakai tabel D1 broker (candle Minggu 2 jam).
#
# SATU-SATUNYA PERUBAHAN STRUKTURAL (behavior-preserving)
#   Perhitungan crossover dipindah KE DALAM helper (diturunkan dari level_col +
#   side). Predikatnya IDENTIK dengan versi terpisah; hanya lokasinya dipindah
#   supaya satu helper melayani 4 level tanpa menyalin kolom crossover manual
#   (prinsip anti-T1). Deteksi tetap pakai >/< (ketat); pelabelan pakai <=/>=.
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ------------------------------------------------------------------------------
# KONFIGURASI
# ------------------------------------------------------------------------------
N_WINDOW        = 6                     # panjang jendela evaluasi (candle H1)
EARLY_K         = [1, 2]                # candle awal untuk deteksi "returned early"
TOL_HOURS       = 8                     # ambang T3 jendela kontinu
START_DATE      = "2016-01-01"
OUTCOME_CLASSES = ["IMMEDIATE_SWEEP", "DELAYED_SWEEP", "FAILED_SWEEP", "PURE_BREAKOUT"]

SESSION_TZ          = "America/New_York"
SESSION_SHIFT_HOURS = 7                 # 24 - 17; sesi dilabeli dengan tanggal SELESAI-nya

# Determinisme: timestamp text tanpa tz -> paksa sesi Spark ke UTC (data verified UTC).
spark.conf.set("spark.sql.session.timeZone", "UTC")


def add_session_date(df, ts_col="timestamp"):
    """UTC -> waktu NY -> geser +7 jam -> ambil tanggal.

    from_utc_timestamp menangani DST otomatis: batas hari bergeser sendiri antara
    22:00 UTC (US DST) dan 23:00 UTC (di luar itu). Arah +7 (bukan -17) memastikan
    sesi Minggu-malam NY -> Senin dilabeli tanggal SELESAI-nya (Senin), bukan Minggu.
    """
    ny = F.from_utc_timestamp(F.col(ts_col), SESSION_TZ)
    return df.withColumn(
        "session_date",
        F.to_date(ny + F.expr(f"INTERVAL {SESSION_SHIFT_HOURS} HOURS"))
    )


# ==============================================================================
# STEP 1: LOAD H1 (SEKALI) + SESSION_DATE + TRIPWIRE T13 + LABEL SESI
# ==============================================================================
# Tabel D1 broker TIDAK dimuat: batas harinya kalender UTC sehingga memuat "candle
# Minggu" 2 jam yang justru jadi sumber masalah. Level diagregasi ulang dari H1.
df_h1_raw = spark.table("market.default.xauusd_h_1")

# Prasyarat determinisme: orderBy("timestamp") bukan urutan total bila ada timestamp
# duplikat -> lead() dan row_number() jadi arbitrer & berubah antar-run. Harus nol.
_dup_h1 = df_h1_raw.groupBy("Time").count().filter(F.col("count") > 1).count()
assert _dup_h1 == 0, f"H1 punya {_dup_h1} timestamp duplikat - lead/row_number non-deterministik"

df_h1 = df_h1_raw.withColumnRenamed("Time", "timestamp") \
                 .filter(F.col("timestamp") >= START_DATE) \
                 .withColumn("timestamp", F.to_timestamp("timestamp"))
df_h1 = add_session_date(df_h1)

# --- TRIPWIRE T13: batas hari wajib menghasilkan hari perdagangan yang utuh ---
_cnt = df_h1.groupBy("session_date").count()
assert _cnt.filter(F.col("count") < 12).count() == 0, \
    "FATAL: ada hari perdagangan <12 candle H1 - batas sesi tidak benar"
assert _cnt.filter(F.dayofweek("session_date").isin(1, 7)).count() == 0, \
    "FATAL: ada hari perdagangan jatuh di Sabtu/Minggu"
print(f"[OK] {_cnt.count()} hari perdagangan, minimum {_cnt.agg(F.min('count')).first()[0]} candle/hari")

# --- T4.b: LABEL SESI SADAR ZONA WAKTU ---
# Urutan when() menetapkan prioritas saat sesi tumpang-tindih:
# LONDON menang atas NY pada jam overlap 13:00-16:00 London.
df_h1 = (df_h1
    .withColumn("hour_ny",     F.hour(F.from_utc_timestamp("timestamp", "America/New_York")))
    .withColumn("hour_london", F.hour(F.from_utc_timestamp("timestamp", "Europe/London")))
    .withColumn("hour_tokyo",  F.hour(F.from_utc_timestamp("timestamp", "Asia/Tokyo")))
    .withColumn("session",
        F.when((F.col("hour_tokyo")  >= 9) & (F.col("hour_tokyo")  < 15), "ASIA")
         .when((F.col("hour_london") >= 8) & (F.col("hour_london") < 16), "LONDON")
         .when((F.col("hour_ny")     >= 8) & (F.col("hour_ny")     < 17), "NY")
         .otherwise("OFF_SESSION")))

# ==============================================================================
# STEP 1b: KUNCI MINGGU (T12) + 3 TRIPWIRE MINGGU
# ==============================================================================
# T12: kunci lama F.year()+F.weekofyear() mencampur tahun kalender & minggu ISO
# -> event membengkak (632 vs 427). date_trunc("week", session_date) tunggal,
# monoton, anti-tabrakan. Karena diturunkan dari session_date (sudah +7jam NY),
# pembuka pekan Minggu-malam OTOMATIS masuk ke minggu yang benar.
df_h1 = df_h1.withColumn("week_start", F.date_trunc("week", F.col("session_date")))

# Tripwire 1: tiap kunci minggu wajib merentang <= 7 hari.
_span = df_h1.groupBy("week_start").agg(
    F.datediff(F.max("session_date"), F.min("session_date")).alias("span_days"))
_bad_span = _span.filter(F.col("span_days") > 7).count()
assert _bad_span == 0, (
    f"FATAL: {_bad_span} kunci minggu merentang >7 hari. "
    "Kunci minggu tidak koheren - jangan lanjutkan.")

# Tripwire 2: awal minggu wajib Senin (konsekuensi date_trunc("week")).
assert df_h1.filter(F.dayofweek("week_start") != 2).count() == 0, \
    "FATAL: ada week_start bukan hari Senin"

# Tripwire 3 (T12c): candle ber-stamp Minggu (UTC, pembuka pekan) wajib terlabeli
# sesi Senin lewat session_date. Robust terhadap hari libur.
_sun = df_h1.filter(F.dayofweek(F.col("timestamp")) == 1)              # Spark: Sunday = 1
_n_bad_sun = _sun.filter(F.dayofweek(F.col("session_date")) != 2).count()  # Monday = 2
assert _n_bad_sun == 0, (
    f"FATAL: {_n_bad_sun} candle hari Minggu tidak terlabeli sesi Senin - T12c belum aktif.")
print(f"[OK] {_span.count()} kunci pekan, span <=7 hari, mulai Senin; "
      f"{_sun.count()} candle Minggu semuanya terlabeli Senin")

# ==============================================================================
# STEP 2: LEAD (SEKALI, SEBELUM JOIN)
# ==============================================================================
# lead() atas SATU window terurut kontinu. Dihitung sekali di sini supaya baris
# yang mungkin ter-drop saat join TIDAK menggeser lead. Dipakai keempat level.
window_time = Window.orderBy("timestamp")

df_h1_lead = df_h1
for k in range(1, N_WINDOW + 1):
    df_h1_lead = df_h1_lead.withColumn(f"close_lead_{k}", F.lead("close", k).over(window_time))
    df_h1_lead = df_h1_lead.withColumn(f"high_lead_{k}",  F.lead("high",  k).over(window_time))
    df_h1_lead = df_h1_lead.withColumn(f"low_lead_{k}",   F.lead("low",   k).over(window_time))

# T3: timestamp candle ke-N untuk mengukur rentang waktu SESUNGGUHNYA.
df_h1_lead = df_h1_lead.withColumn("ts_lead_N", F.lead("timestamp", N_WINDOW).over(window_time))

# ==============================================================================
# STEP 3: AGREGASI LEVEL DARI H1 (DAILY: PDH/PDL, WEEKLY: PWH/PWL)
# ==============================================================================
# T13: level diagregasi ULANG dari H1 (bukan tabel D1/W-1 broker).
# PDH/PDL = high/low hari SEBELUMNYA (D-1) lewat lag(1) atas urutan session_date.
df_daily_levels = (df_h1.groupBy("session_date")
    .agg(F.max("high").alias("d_high"), F.min("low").alias("d_low")))
_wd = Window.orderBy("session_date")
df_daily_levels = (df_daily_levels
    .withColumn("PDH", F.lag("d_high", 1).over(_wd))
    .withColumn("PDL", F.lag("d_low",  1).over(_wd))
    .select("session_date", "PDH", "PDL"))

# PWH/PWL = high/low pekan SEBELUMNYA (W-1) lewat lag(1) atas urutan week_start.
df_weekly_levels = (df_h1.groupBy("week_start")
    .agg(F.max("high").alias("weekly_high"), F.min("low").alias("weekly_low")))
_ww = Window.orderBy("week_start")
df_weekly_levels = (df_weekly_levels
    .withColumn("PWH", F.lag("weekly_high", 1).over(_ww))
    .withColumn("PWL", F.lag("weekly_low",  1).over(_ww))
    .select("week_start", "PWH", "PWL"))

# ==============================================================================
# STEP 4: JOIN LEVEL KE H1 (LEFT, T6) + PREV HIGH/LOW UNTUK CROSSOVER
# ==============================================================================
# LEFT join (T6): candle tanpa pasangan level tidak lenyap diam-diam, tetap ada
# dan tercatat di funnel. Baris tanpa level tidak akan pernah lolos syarat crossover.
n_before_join = df_h1_lead.count()
df_h1_silver = df_h1_lead.join(df_daily_levels,  on="session_date", how="left") \
                         .join(df_weekly_levels, on="week_start",   how="left")
n_orphan_daily  = df_h1_silver.filter(F.col("PDH").isNull()).count()
n_orphan_weekly = df_h1_silver.filter(F.col("PWH").isNull()).count()
print(f"[FUNNEL] baris H1                       : {n_before_join}")
print(f"[FUNNEL] baris tanpa level D-1 (orphan) : {n_orphan_daily}")
print(f"[FUNNEL] baris tanpa level W-1 (orphan) : {n_orphan_weekly}")

# prev_high/prev_low level-independent -> dihitung SEKALI, dipakai keempat level.
# NOTE: lead sudah dihitung di STEP 2, SEBELUM join. Jangan pindahkan ke sini.
df_h1_silver = df_h1_silver \
    .withColumn("prev_high", F.lag("high", 1).over(window_time)) \
    .withColumn("prev_low",  F.lag("low",  1).over(window_time))


# ==============================================================================
# HELPER PELABELAN - SATU definisi untuk keempat level (prinsip anti-T1)
# ==============================================================================
def label_liquidity_events(df, level_col, side, n_window, tol_hours, level_name,
                           partition_cols, funnel):
    """side: 'high' (PDH/PWH, pakai <=) atau 'low' (PDL/PWL, pakai >=).

    Crossover dihitung DI DALAM helper dari level_col + side. Predikatnya IDENTIK
    dengan versi terpisah (high: high>level & prev_high<=level; low: low<level &
    prev_low>=level) - hanya lokasinya dipindah supaya satu helper melayani 4 level
    tanpa menyalin kolom crossover manual. Deteksi ketat (>/<); pelabelan inklusif.
    """
    # --- (4) tandai crossover (deteksi ketat)
    if side == "high":
        cross = (F.col("high") > F.col(level_col)) & (F.col("prev_high") <= F.col(level_col))
    else:
        cross = (F.col("low") < F.col(level_col)) & (F.col("prev_low") >= F.col(level_col))

    def inside(col):
        """Apakah harga berada di DALAM range relatif level? (pelabelan inklusif)"""
        return F.col(col) <= F.col(level_col) if side == "high" else F.col(col) >= F.col(level_col)

    # --- (5) filter ke baris crossover saja
    ev = df.filter(cross)
    funnel[f"{level_name}_1_crossovers_all"] = ev.count()

    # --- (6) LEVEL EXHAUSTION: hanya sentuhan pertama per periode
    w = Window.partitionBy(*partition_cols).orderBy("timestamp")
    ev = ev.withColumn("touch_rank", F.row_number().over(w)).filter(F.col("touch_rank") == 1)
    n_exh = ev.count()
    funnel[f"{level_name}_2_after_exhaustion"] = n_exh

    # --- (7) T2: buang jendela tidak lengkap. WAJIB SETELAH LANGKAH (6).
    #     Kalau dipasang SEBELUM row_number(), sentuhan kedua di periode yang sama
    #     akan NAIK PANGKAT jadi rank 1 - aturan level exhaustion jebol diam-diam.
    #     Cek kolom lead TERAKHIR sudah cukup: satu window terurut kontinu menjamin
    #     close_lead_N tidak null => lead_1..N-1 juga tidak null.
    ev = ev.filter(F.col(f"close_lead_{n_window}").isNotNull())
    n_complete = ev.count()
    funnel[f"{level_name}_3_dropped_T2"] = n_exh - n_complete
    funnel[f"{level_name}_4_after_T2"] = n_complete

    # --- (8) T3: penanda kontinuitas jendela (PENANDA, bukan filter)
    ev = ev.withColumn(
        "window_hours",
        (F.unix_timestamp("ts_lead_N") - F.unix_timestamp("timestamp")) / 3600
    ).withColumn(
        "window_is_continuous", F.when(F.col("window_hours") <= tol_hours, 1).otherwise(0)
    )

    # --- (9) T1: dua predikat dasar, lalu 4 kelas SALING LEPAS
    returned_early = inside(f"close_lead_{EARLY_K[0]}")
    for k in EARLY_K[1:]:
        returned_early = returned_early | inside(f"close_lead_{k}")

    ev = ev.withColumn("returned_early", returned_early) \
           .withColumn("ended_inside", inside(f"close_lead_{n_window}"))

    # Keempat kondisi EKSPLISIT. Tidak ada .otherwise(PURE_BREAKOUT) - itu akan
    # mengulang pola kegagalan T2: null apa pun yang lolos filter diam-diam dilabeli
    # PURE_BREAKOUT. Cabang terakhir sengaja NULL sebagai sentinel, lalu di-assert nol.
    ev = ev.withColumn(
        "outcome",
        F.when(F.col("returned_early") & F.col("ended_inside"), F.lit("IMMEDIATE_SWEEP"))
         .when(~F.col("returned_early") & F.col("ended_inside"), F.lit("DELAYED_SWEEP"))
         .when(F.col("returned_early") & ~F.col("ended_inside"), F.lit("FAILED_SWEEP"))
         .when(~F.col("returned_early") & ~F.col("ended_inside"), F.lit("PURE_BREAKOUT"))
         .otherwise(F.lit(None))
    )

    # Kolom turunan DITURUNKAN SECARA MEKANIS - derivasi, bukan duplikasi.
    # Dikunci assertion sum(one-hot)==1 di bawah.
    ev = ev.withColumn("is_sweep", F.col("ended_inside").cast("int"))
    for c in OUTCOME_CLASSES:
        ev = ev.withColumn(f"is_{c.lower()}", (F.col("outcome") == c).cast("int"))

    return ev.withColumn("level_type", F.lit(level_name)) \
             .withColumn("level_price", F.col(level_col))


# ==============================================================================
# STEP 5: PANGGIL HELPER 4x (PDH/PDL by session_date; PWH/PWL by week_start) + UNION
# ==============================================================================
funnel = {}

df_pdh = label_liquidity_events(
    df_h1_silver, level_col="PDH", side="high", n_window=N_WINDOW,
    tol_hours=TOL_HOURS, level_name="PDH", partition_cols=["session_date"], funnel=funnel)

df_pdl = label_liquidity_events(
    df_h1_silver, level_col="PDL", side="low", n_window=N_WINDOW,
    tol_hours=TOL_HOURS, level_name="PDL", partition_cols=["session_date"], funnel=funnel)

df_pwh = label_liquidity_events(
    df_h1_silver, level_col="PWH", side="high", n_window=N_WINDOW,
    tol_hours=TOL_HOURS, level_name="PWH", partition_cols=["week_start"], funnel=funnel)

df_pwl = label_liquidity_events(
    df_h1_silver, level_col="PWL", side="low", n_window=N_WINDOW,
    tol_hours=TOL_HOURS, level_name="PWL", partition_cols=["week_start"], funnel=funnel)

df_gold = df_pdh.unionByName(df_pdl).unionByName(df_pwh).unionByName(df_pwl) \
               .orderBy("timestamp").cache()

# ==============================================================================
# STEP 6: VALIDASI - bagian permanen pipeline
# ==============================================================================
ONEHOTS = [f"is_{c.lower()}" for c in OUTCOME_CLASSES]

# (a) tepat satu one-hot menyala (membuktikan label WELL-FORMED).
_k = df_gold.withColumn("k", sum(F.col(c) for c in ONEHOTS))
assert _k.filter((F.col("k") != 1) | F.col("k").isNull()).count() == 0, \
    "FATAL: one-hot tidak berjumlah tepat 1 - derivasi melenceng dari outcome"

# (b) domain tertutup + deteksi fallthrough.
assert df_gold.filter(F.col("outcome").isNull()).count() == 0, \
    "FATAL: ada event tanpa klasifikasi - sentinel menyala, ada NULL yang lolos"
assert df_gold.filter(~F.col("outcome").isin(*OUTCOME_CLASSES)).count() == 0, \
    "FATAL: nilai outcome di luar domain"
assert df_gold.filter(
    F.col("returned_early").isNull() | F.col("ended_inside").isNull()).count() == 0, \
    "FATAL: predikat NULL - prasyarat pelabelan tidak terpenuhi"

# (c) penurunan ulang semantik dari harga (membuktikan label BENAR).
#     high-side (PDH/PWH) berakhir di luar => close_lead_N > level; low-side => <.
_recomputed = F.when(
    F.col("level_type").isin("PDH", "PWH"), F.col(f"close_lead_{N_WINDOW}") > F.col("level_price")
).otherwise(F.col(f"close_lead_{N_WINDOW}") < F.col("level_price"))
assert df_gold.filter(
    _recomputed != F.col("outcome").isin("PURE_BREAKOUT", "FAILED_SWEEP")).count() == 0, \
    "FATAL: outcome tidak konsisten dengan harga akhir jendela"

# (d) invarian level exhaustion: PDH/PDL per (session_date, level), PWH/PWL per (week_start, level).
assert df_gold.filter(F.col("level_type").isin("PDH", "PDL")) \
    .groupBy("session_date", "level_type").count().filter(F.col("count") > 1).count() == 0, \
    "FATAL: ada >1 event per (hari, level) - level exhaustion jebol (daily)"
assert df_gold.filter(F.col("level_type").isin("PWH", "PWL")) \
    .groupBy("week_start", "level_type").count().filter(F.col("count") > 1).count() == 0, \
    "FATAL: ada >1 event per (minggu, level) - level exhaustion jebol (weekly)"

# --- TABEL KONTINGENSI: definisi label degenerate muncul sebagai SEL KOSONG ---
print("\n=== Kontingensi returned_early x ended_inside (keempat sel WAJIB terisi) ===")
df_gold.groupBy("returned_early", "ended_inside").count().orderBy(
    "returned_early", "ended_inside").show()
_empty = 4 - df_gold.select("returned_early", "ended_inside").distinct().count()
assert _empty == 0, f"FATAL: {_empty} sel kontingensi kosong - definisi label degenerate"

print("=== Distribusi outcome per level ===")
df_gold.groupBy("level_type", "outcome").count().orderBy("level_type", "outcome").show()

print("=== Sweep rate per level (is_sweep = ended_inside) ===")
df_gold.groupBy("level_type").agg(
    F.count("*").alias("n"),
    F.round(F.avg("is_sweep") * 100, 2).alias("sweep_rate_pct")).orderBy("level_type").show()

print("=== Funnel kualitas data ===")
for k in sorted(funnel):
    print(f"  {k:34s} {funnel[k]:>6}")
_total = df_gold.count()
_disc = df_gold.filter(F.col("window_is_continuous") == 0).count()
print(f"  {'5_total_gold':34s} {_total:>6}")
print(f"  {'6_jendela_terputus_T3':34s} {_disc:>6}  ({100*_disc/_total:.1f}%)")
print("\n  -> ANALISIS UTAMA memakai window_is_continuous == 1")
print("  -> event berjendela terputus dilaporkan TERPISAH sebagai sub-temuan")

# ==============================================================================
# STEP 7: SIMPAN - satu tabel gold + CSV + funnel
# ==============================================================================
# Simpan kolom audit (returned_early, ended_inside, close/high/low_lead_*, window_hours)
# supaya setiap label bisa DITURUNKAN ULANG DENGAN TANGAN baris demi baris saat sidang.
df_gold.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("market.default.xauusd_liquidity_gold")

df_gold.coalesce(1).write.option("header", "true").mode("overwrite") \
    .csv("/Volumes/market/default/market/output_liquidity.csv")

spark.createDataFrame(
    [(k, int(v)) for k, v in sorted(funnel.items())], ["tahap", "jumlah"]
).write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("market.default.xauusd_liquidity_dq_funnel")

display(df_gold.select(
    "timestamp", "session_date", "week_start", "session", "level_type", "level_price", "close",
    "returned_early", "ended_inside", "outcome", "is_sweep",
    "window_hours", "window_is_continuous"))

df_gold.unpersist()
